# Diffusion through Fourier modes in Julia

I calculate diffusion on a periodic line. All quantities in this example are dimensionless: position x, time t, concentration c and diffusivity D. The domain length is 2π and D = 1. Fourier wave number k identifies a spatial mode. I use exact propagation of each mode rather than an approximate time-stepping scheme.

Run each small part and its checks in order. Julia indices start at 1. For Fourier examples install FFTW with `julia -e 'using Pkg; Pkg.add("FFTW")'`. For notebook use, install IJulia and select its kernel. The script is also supplied. Julia and NumPy random sequences differ; compare ensemble statistics, not individual paths.

## 1. Define the grid

I omit the repeated endpoint of the periodic interval. I check that the grid spacing times the number of points equals the domain length.

In [ ]:
using FFTW, Statistics
n = 128
length_x = 2π
spacing = length_x / n
x = spacing .* (0:n-1)
k = 2π .* fftfreq(n, 1 / spacing)
diffusivity = 1.0
@assert isapprox(n * spacing, length_x)
@assert k[1] == 0
println("Grid points: ", n, "; spacing: ", spacing)

## 2. Write the initial concentration

The constant term sets the mean. Two cosine terms give two spatial wavelengths. I check the mean and transform the field back before introducing evolution.

In [ ]:
initial = 1 .+ 0.4 .* cos.(x) .+ 0.2 .* cos.(4 .* x)
spectrum = fft(initial)
@assert abs(mean(initial) - 1) < 1e-12
@assert maximum(abs.(real.(ifft(spectrum)) .- initial)) < 1e-12
println("Initial mean: ", mean(initial))

## 3. Evolve one Fourier mode

Diffusion multiplies a Fourier mode by exp(-D k² t). Higher wave numbers decay faster. I check the known concentration profile at one time; this check would catch a missing square on k.

In [ ]:
time = 0.3
decay = exp.(-diffusivity .* k.^2 .* time)
field = real.(ifft(spectrum .* decay))
exact = 1 .+ 0.4 * exp(-time) .* cos.(x) .+ 0.2 * exp(-16*time) .* cos.(4 .* x)
@assert maximum(abs.(field .- exact)) < 1e-12
@assert abs(mean(field) - mean(initial)) < 1e-12
println("Maximum analytical error: ", maximum(abs.(field .- exact)))

## 4. Repeat at several times

I keep the same initial spectrum for every time, since each evaluation gives the solution from the initial state. I check mean conservation and the decreasing concentration variation. This is not a time-step convergence study.

In [ ]:
times = [0.0, 0.02, 0.1, 0.3, 1.0]
profiles = Vector{Float64}[]
variation = Float64[]
for sample_time in times
    sample_decay = exp.(-diffusivity .* k.^2 .* sample_time)
    profile = real.(ifft(spectrum .* sample_decay))
    push!(profiles, profile)
    push!(variation, mean((profile .- mean(profile)).^2))
end
@assert all(abs(mean(profile) - 1) < 1e-12 for profile in profiles)
@assert all(diff(variation) .<= 0)
println("Concentration variances: ", variation)